# Partie 1 — Bronze

In [1]:
# 1. Créer la session Spark de ce notebook
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse-Partie1-Bronze")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.medical", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.medical.type", "rest")
    .config("spark.sql.catalog.medical.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.medical.warehouse", "s3://medical-warehouse/")
    .config("spark.sql.catalog.medical.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.medical.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.medical.s3.path-style-access", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Version de Spark :", spark.version)
print("Catalogue :", spark.conf.get("spark.sql.catalog.medical"))
print("REST :", spark.conf.get("spark.sql.catalog.medical.uri"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 09:50:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 09:50:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/19 09:50:49 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/10/19 09:50:49 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Version de Spark : 3.5.5
Catalogue : org.apache.iceberg.spark.SparkCatalog
REST : http://iceberg-rest:8181


In [2]:
# 2. Vérifier que Spark voit le catalogue
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

+---------+
|namespace|
+---------+
|sandbox  |
|bronze   |
+---------+



In [3]:
# 3. Vérifier les chemins des fichiers
from pathlib import Path

DATA = Path("/workspace/notebooks/data")
SYNTHEA = DATA / "synthea"
PREPARED = DATA / "prepared"

patients_csv = SYNTHEA / "patients.csv"   # archive d'origine (non utilisée pour Bronze)
consultations_csv = SYNTHEA / "encounters.csv"
observations_csv = SYNTHEA / "observations.csv"
observations_json = PREPARED / "observations.jsonl"

for chemin in [patients_csv, consultations_csv, observations_csv]:
    print(chemin.name, "→", "OK" if chemin.is_file() else "INTROUVABLE")
    if not chemin.is_file():
        raise FileNotFoundError(f"Fichier manquant : {chemin}")

PREPARED.mkdir(parents=True, exist_ok=True)

patients.csv → OK
encounters.csv → OK
observations.csv → OK


## 4 — Préparer `observations.jsonl`

Synthea fournit les observations en CSV. Pour simuler une troisième source, nous créons **un objet JSON par ligne** en conservant les valeurs d'origine. Cette préparation ne constitue pas un nettoyage Silver.

In [4]:
# 4. Convertir le CSV d'observations en JSON Lines
import csv
import json

nombre_converti = 0

with observations_csv.open("r", encoding="utf-8-sig", newline="") as fichier_csv:
    lecteur = csv.DictReader(fichier_csv)
    with observations_json.open("w", encoding="utf-8") as fichier_json:
        for ligne in lecteur:
            fichier_json.write(json.dumps(ligne, ensure_ascii=False) + "\n")
            nombre_converti += 1

print("Fichier JSON créé :", observations_json)
print("Nombre d'observations converties :", nombre_converti)

Fichier JSON créé : /workspace/notebooks/data/prepared/observations.jsonl
Nombre d'observations converties : 68648


## 5 — Lire les patients depuis PostgreSQL

Nous ne réutilisons **aucune variable** d'un autre notebook : la connexion JDBC et le DataFrame sont créés ici.

À l'intérieur de Docker, PostgreSQL est accessible sur `postgres:5432`, et non sur `localhost:5433`.

In [5]:
# 5. Paramètres de connexion à PostgreSQL
jdbc_options = {
    "url": "jdbc:postgresql://postgres:5432/hospital",
    "dbtable": "public.patients_source",
    "user": "user",
    "password": "0000",
    "driver": "org.postgresql.Driver"
}

# Lire les patients directement depuis PostgreSQL
df_patients = (
    spark.read
    .format("jdbc")
    .options(**jdbc_options)
    .load()
)

df_patients.printSchema()
df_patients.show(5, truncate=False)
nb_patients = df_patients.count()
print("Patients PostgreSQL :", nb_patients)
assert nb_patients > 0, "La table PostgreSQL patients_source est vide."

root
 |-- patient_id: string (nullable = true)
 |-- birthdate: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)

+------------------------------------+----------+------+----------------+
|patient_id                          |birthdate |gender|city            |
+------------------------------------+----------+------+----------------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16|M     |North Brookfield|
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11|F     |Bedford         |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15|F     |Quincy          |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19|F     |Taunton         |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23|M     |Boston          |
+------------------------------------+----------+------+----------------+
only showing top 5 rows

Patients PostgreSQL : 108


## 6 — Lire les consultations depuis le CSV

Nous conservons les colonnes et les valeurs sources. Les dates restent du texte dans Bronze.

In [6]:
# 6. Lire les consultations depuis encounters.csv
df_consultations = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(consultations_csv))
)

df_consultations.printSchema()
df_consultations.show(5, truncate=False)
nb_consultations = df_consultations.count()
print("Consultations CSV :", nb_consultations)
assert nb_consultations > 0, "Le fichier encounters.csv semble vide."

root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)

+------------------------------------+--------------------+--------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+---------+------------------------------------------+-------------------+----------------+--------------+----------+-------

## 7 — Lire les observations depuis le JSON

Spark lit les objets JSON ligne par ligne et construit un DataFrame. Nous ne transformons pas les unités ni les valeurs médicales à cette étape.

In [7]:
# 7. Lire les observations en JSON
df_observations = spark.read.json(str(observations_json))

df_observations.printSchema()
df_observations.show(5, truncate=False)
nb_observations = df_observations.count()
print("Observations JSON :", nb_observations)
assert nb_observations == nombre_converti, "Le nombre de lignes JSON est différent du CSV."

root
 |-- CATEGORY: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- TYPE: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- VALUE: string (nullable = true)

+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|CATEGORY   |CODE   |DATE                |DESCRIPTION                                                  |ENCOUNTER                           |PATIENT                             |TYPE   |UNITS  |VALUE|
+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|laboratory |4548-4 |2018-01-22T08:35:36Z|Hemo

## 8 — Vérifier les trois sources avant l'ingestion

Les quantités affichées servent de référence : Bronze devra contenir **autant de lignes que les sources**.

In [8]:
# 8. Récapitulatif des sources
print("=== SOURCES ===")
print("Patients PostgreSQL :", nb_patients)
print("Consultations CSV   :", nb_consultations)
print("Observations JSON   :", nb_observations)

=== SOURCES ===
Patients PostgreSQL : 108
Consultations CSV   : 5571
Observations JSON   : 68648


In [ ]:
###ici on commence le traitement

## 9 — Créer le namespace `bronze`

Le nom `medical.bronze.patients` signifie : **catalogue `medical` → namespace `bronze` → table `patients`**.

`IF NOT EXISTS` permet de relancer la cellule sans erreur si le namespace a déjà été créé.

In [9]:
# 9. Créer l'espace Bronze dans le catalogue
spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.bronze")
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

+---------+
|namespace|
+---------+
|sandbox  |
|bronze   |
+---------+



## 10 — Écrire la table Bronze des patients

Spark copie les lignes de PostgreSQL dans une **table Iceberg**. Iceberg référence des fichiers de données et leurs métadonnées dans RustFS.

Si la table existe déjà, nous **ne l'écrasons pas** et ne dupliquons pas les lignes.

In [10]:
# 10. PostgreSQL -> Iceberg Bronze patients
nom_table = "medical.bronze.patients"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_patients.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Patients dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

Table créée : medical.bronze.patients
Patients dans Bronze : 108
+------------------------------------+----------+------+----------------+
|patient_id                          |birthdate |gender|city            |
+------------------------------------+----------+------+----------------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16|M     |North Brookfield|
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11|F     |Bedford         |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15|F     |Quincy          |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19|F     |Taunton         |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23|M     |Boston          |
+------------------------------------+----------+------+----------------+
only showing top 5 rows



## 11 — Écrire la table Bronze des consultations

Même opération, mais cette fois les données viennent d'un fichier CSV. Les champs d'origine sont conservés.

In [11]:
# 11. CSV -> Iceberg Bronze consultations
nom_table = "medical.bronze.consultations"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_consultations.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Consultations dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

Table créée : medical.bronze.consultations
Consultations dans Bronze : 5571
+------------------------------------+--------------------+--------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+---------+------------------------------------------+-------------------+----------------+--------------+----------+-----------------+
|Id                                  |START               |STOP                |PATIENT                             |ORGANIZATION                        |PROVIDER                            |PAYER                               |ENCOUNTERCLASS|CODE     |DESCRIPTION                               |BASE_ENCOUNTER_COST|TOTAL_CLAIM_COST|PAYER_COVERAGE|REASONCODE|REASONDESCRIPTION|
+------------------------------------+--------------------+--------------------+------------------------------------+------------------------------------+--------------

## 12 — Écrire la table Bronze des observations

Les observations JSON deviennent à leur tour une table Iceberg. Nous ne nettoyons pas encore les valeurs.

In [12]:
# 12. JSON -> Iceberg Bronze observations
nom_table = "medical.bronze.observations"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_observations.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Observations dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

Table créée : medical.bronze.observations
Observations dans Bronze : 68648
+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|CATEGORY   |CODE   |DATE                |DESCRIPTION                                                  |ENCOUNTER                           |PATIENT                             |TYPE   |UNITS  |VALUE|
+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|laboratory |4548-4 |2018-01-22T08:35:36Z|Hemoglobin A1c/Hemoglobin.total in Blood                     |cc3eac4a-34a7-9a15-da83-de2c625ae978|cc3eac4a-34a7-9a15-1522-8087135631b7|numeric|%      |6.3  |
|vital-signs|8302-2 |2018-01-22T08:35:36Z|Body Height                                                  |cc3eac4a-34a7-9a1

## 13 — Afficher les tables du catalogue

`SHOW TABLES` vérifie que le catalogue Iceberg connaît nos trois tables Bronze.

In [13]:
# 13. Lister les tables de la couche Bronze
spark.sql("SHOW TABLES IN medical.bronze").show(truncate=False)

+---------+-------------+-----------+
|namespace|tableName    |isTemporary|
+---------+-------------+-----------+
|bronze   |consultations|false      |
|bronze   |observations |false      |
|bronze   |patients     |false      |
+---------+-------------+-----------+



## 14 — Vérifier le nombre de lignes

Bronze doit conserver toutes les lignes. Si une table existait déjà avec un autre contenu, un message **ATTENTION** s'affichera : elle ne sera pas modifiée automatiquement.

In [14]:
# 14. Comparer les nombres de lignes sources et Bronze
comparaison = [
    ("patients", nb_patients),
    ("consultations", nb_consultations),
    ("observations", nb_observations)
]

for nom, nb_source in comparaison:
    nb_bronze = spark.table(f"medical.bronze.{nom}").count()
    etat = "OK" if nb_source == nb_bronze else "ATTENTION : nombres différents"
    print(f"{nom} : source={nb_source}, bronze={nb_bronze} -> {etat}")

patients : source=108, bronze=108 -> OK
consultations : source=5571, bronze=5571 -> OK
observations : source=68648, bronze=68648 -> OK


## 15 — Interroger Bronze avec Spark SQL

Les données sont maintenant consultables comme des tables SQL, même si elles sont physiquement stockées dans RustFS.

In [15]:
# 15. Exemple de requête SQL sur Bronze
spark.sql("""
    SELECT patient_id, birthdate, gender, city
    FROM medical.bronze.patients
    LIMIT 5
""").show(truncate=False)

+------------------------------------+----------+------+----------------+
|patient_id                          |birthdate |gender|city            |
+------------------------------------+----------+------+----------------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16|M     |North Brookfield|
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11|F     |Bedford         |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15|F     |Quincy          |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19|F     |Taunton         |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23|M     |Boston          |
+------------------------------------+----------+------+----------------+



## 16 — Voir les fichiers de la table Iceberg

La table de métadonnées `.files` contient notamment le chemin de chaque fichier de données et son nombre de lignes. Vous devriez voir des chemins vers le bucket `medical-warehouse`.

In [16]:
# 16. Afficher les fichiers de données Iceberg
spark.sql("""
    SELECT file_path, record_count, file_size_in_bytes
    FROM medical.bronze.patients.files
""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                |record_count|file_size_in_bytes|
+---------------------------------------------------------------------------------------------------------+------------+------------------+
|s3://medical-warehouse/bronze/patients/data/00000-13-09d2e758-487d-4168-ae6f-30ffb8849f70-0-00001.parquet|108         |4837              |
+---------------------------------------------------------------------------------------------------------+------------+------------------+



## 17 — Voir les snapshots Iceberg

Un **snapshot** correspond à un état enregistré de la table. Iceberg utilise ces snapshots pour suivre les changements de données.

In [17]:
# 17. Consulter l'historique des snapshots
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM medical.bronze.patients.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|3176723268370757061|2026-10-19 09:51:53.934|append   |
+-------------------+-----------------------+---------+

